# Analyse African Air Quality Data with AirQo and Python

**Goal:** Explore PM₂.₅ levels across African cities using the AirQo network,
compare against WHO guidelines, and assess data quality from low-cost sensors
in a data-sparse region.

**API keys required:** `AIRQO_API_KEY`

**Aeolus features demonstrated:**
- `networks.get_metadata("AIRQO")` for sensor discovery
- `download("AIRQO", ...)` for African network data
- `metrics.aqi_check_who()` for WHO compliance
- `metrics.time_average()` for daily/monthly aggregation
- `viz.plot_distribution()` for cross-city comparison
- `viz.plot_monthly()` for seasonal patterns

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("AIRQO_API_KEY"):
    raise EnvironmentError(
        "AIRQO_API_KEY is required for this notebook.\n"
        "Request access at https://www.airqo.net/\n"
        "Then add it to your .env file: AIRQO_API_KEY=your_key_here"
    )

In [2]:
import aeolus
from aeolus import metrics, viz
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

## 1. Explore the AirQo Network

AirQo operates 200+ low-cost sensors across African cities, primarily in
East Africa. Let's explore what's available.

In [3]:
# Get network information
info = aeolus.get_source_info("AIRQO")
print(f"Source: {info['name']}")
print(f"Type: {info['type']}")

Source: AirQo
Type: network


In [4]:
# Fetch all AirQo metadata
metadata = aeolus.networks.get_metadata("AIRQO")

print(f"Total AirQo sensors: {len(metadata)}")
print(f"\nColumns: {metadata.columns.tolist()}")
print(f"\nSample sites:")
metadata[["site_code", "site_name", "latitude", "longitude"]].head(10)

Total AirQo sensors: 531

Columns: ['site_code', 'site_name', 'generated_name', 'formatted_name', 'approximate_latitude', 'approximate_longitude', 'country', 'region', 'city', 'search_name', 'location_name', 'isOnline', 'rawOnlineStatus', 'lastRawData', 'grid_name', 'grid_id', 'district', 'county', 'sub_county', 'parish', 'division', 'latitude', 'longitude', 'source_network', 'measurands']

Sample sites:


,site_code,site_name,latitude,longitude
0,69419f4609357b00130ab4f7,Malindi Stage Road,-3.221394,40.115212
1,6941a183fe657a0013b41793,Sabaki Junction Lamu Road,-3.215598,40.112854
2,6941a4edfe657a0013b4187a,Tsavo Road,-3.220534,40.113737
3,64ceb28f2d581c00135ccbd7,University of Pretoria,-25.755769,28.226053
4,659727fe88a3390012aca615,South African Weather Station (SAWS),-25.906411,28.210684
5,6936c465b00c580013d2b925,Hammanskraal,-25.380491,28.285875
6,6936c4d6b00c580013d2b9d7,Germiston Clinic,-26.230744,28.190075
7,6936c554b3394000137e0f2c,Sandton Health department,-26.096890,28.061260
8,6936c5b7b00c580013d2bb3e,Copper college,-26.356992,27.979631
9,6936c641d045e2001208c857,Glen Auston,-25.940733,28.148406


In [5]:
# Check for city/grid information in metadata
if "grid_name" in metadata.columns:
    print("Sites per grid/city:")
    print(metadata["grid_name"].value_counts().head(15))
elif "city" in metadata.columns:
    print("Sites per city:")
    print(metadata["city"].value_counts().head(15))
else:
    # Group by approximate location
    print(f"Available metadata columns: {metadata.columns.tolist()}")
    print(f"\nSite name patterns:")
    print(metadata["site_name"].head(20).tolist())

Sites per grid/city:
grid_name
greater_kampala    116
kampala_city        95
nigeria             47
lagos               39
kampala_central     30
banjul              20
nakawa              17
south_africa        17
gambia              16
gauteng             15
mbarara_city        15
rubaga              12
accra               12
mozambique          10
yaounde_city         9
Name: count, dtype: int64


## 2. Select Sites from Multiple Cities

Pick representative sensors from different locations for comparison.
We select a few sites to keep the download manageable.

In [6]:
# Select sites — take a representative sample
# Adjust these based on what's available in your metadata
selected_sites = metadata["site_code"].head(12).tolist()

print(f"Selected {len(selected_sites)} sites for analysis")
selected_meta = metadata[metadata["site_code"].isin(selected_sites)]
selected_meta[["site_code", "site_name", "latitude", "longitude"]]

Selected 12 sites for analysis


,site_code,site_name,latitude,longitude
0,69419f4609357b00130ab4f7,Malindi Stage Road,-3.221394,40.115212
1,6941a183fe657a0013b41793,Sabaki Junction Lamu Road,-3.215598,40.112854
2,6941a4edfe657a0013b4187a,Tsavo Road,-3.220534,40.113737
3,64ceb28f2d581c00135ccbd7,University of Pretoria,-25.755769,28.226053
4,659727fe88a3390012aca615,South African Weather Station (SAWS),-25.906411,28.210684
5,6936c465b00c580013d2b925,Hammanskraal,-25.380491,28.285875
6,6936c4d6b00c580013d2b9d7,Germiston Clinic,-26.230744,28.190075
7,6936c554b3394000137e0f2c,Sandton Health department,-26.096890,28.061260
8,6936c5b7b00c580013d2bb3e,Copper college,-26.356992,27.979631
9,6936c641d045e2001208c857,Glen Auston,-25.940733,28.148406


## 3. Download PM₂.₅ Data

In [7]:
# Download 3 months of data (AirQo API can be slow for long periods)
data = aeolus.download(
    "AIRQO",
    sites=selected_sites,
    start_date=datetime(2024, 7, 1),
    end_date=datetime(2024, 9, 30),
)

print(f"Downloaded {len(data):,} rows")
print(f"Sites with data: {data['site_code'].nunique()}")
print(f"Pollutants: {data['measurand'].unique().tolist()}")

/Users/ruaraidhdobson/Dropbox/Personal/sls/aeolus/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded 0 rows
Sites with data: 0
Pollutants: []


In [8]:
# Filter to PM2.5
pm25 = data[data["measurand"] == "PM2.5"]
print(f"PM\u2082.\u2085 records: {len(pm25):,}")

# Check ratification — AirQo data is unvalidated low-cost sensor data
print(f"\nRatification flags:")
print(pm25["ratification"].value_counts())

PM₂.₅ records: 0

Ratification flags:
Series([], Name: count, dtype: int64)


## 4. Data Quality Assessment

Low-cost sensors in tropical environments need careful quality assessment.
Check data capture rates and basic statistics.

In [9]:
# Daily data capture per site
daily = metrics.time_average(pm25, freq="D")
daily_pm = daily[daily["measurand"] == "PM2.5"]

# Data capture summary
capture = (
    daily_pm
    .groupby("site_code")["data_capture"]
    .mean()
    .sort_values(ascending=False)
)

print("Mean daily data capture by site:")
for site, cap in capture.items():
    status = "\u2705" if cap >= 0.75 else "\u26a0\ufe0f" if cap >= 0.5 else "\u274c"
    print(f"  {status} {site}: {cap:.0%}")

Mean daily data capture by site:


## 5. WHO Guideline Comparison

African cities often have PM\u2082.\u2085 levels far above WHO guidelines.
Compare against all interim targets.

In [10]:
# WHO compliance check
who_result = metrics.aqi_check_who(pm25, target="AQG")

if who_result.empty or "pollutant" not in who_result.columns:
    print("No WHO compliance results (insufficient data or unrecognised pollutants).")
else:
    pm_who = who_result[who_result["pollutant"] == "PM2.5"]

    if not pm_who.empty:
        print("WHO Annual Guideline (5 \u00b5g/m\u00b3) compliance:")
        for _, row in pm_who.iterrows():
            status = "\u2705" if row["meets_guideline"] else "\u274c"
            print(f"  {status} {row['site_code']}: {row['mean_concentration']:.1f} \u00b5g/m\u00b3 "
                  f"({row['exceedance_ratio']:.1f}x guideline)")
    else:
        print("No PM2.5 results in WHO compliance check.")

No WHO compliance results (insufficient data or unrecognised pollutants).


In [11]:
# Check all WHO targets
print("\nCompliance across WHO targets:")
print(f"{'Target':<8} {'Guideline':>10} {'Sites meeting':>15}")
print("-" * 35)

for target in ["AQG", "IT-4", "IT-3", "IT-2", "IT-1"]:
    result = metrics.aqi_check_who(pm25, target=target)
    if result.empty or "pollutant" not in result.columns:
        continue
    pm_rows = result[result["pollutant"] == "PM2.5"]
    if not pm_rows.empty:
        n_meet = pm_rows["meets_guideline"].sum()
        guideline = pm_rows["guideline_value"].iloc[0]
        print(f"{target:<8} {guideline:>8.0f} \u00b5g/m\u00b3 {n_meet:>6} / {len(pm_rows)}")


Compliance across WHO targets:
Target    Guideline   Sites meeting
-----------------------------------


## 6. Temporal Patterns

In [12]:
# Time series
# Fix datetime dtype for matplotlib compatibility
pm25_plot = pm25.copy()
pm25_plot["date_time"] = pd.to_datetime(pm25_plot["date_time"], utc=True)

if not pm25_plot.empty and "PM2.5" in pm25_plot["measurand"].values:
    fig = viz.plot_timeseries(
        pm25_plot,
        pollutants=["PM2.5"],
        guideline=15.0,
        guideline_label="WHO IT-1 (15 µg/m³)",
        title="AirQo PM₂.₅ Across African Sites",
    )
    plt.show()
else:
    print("No PM2.5 data available for timeseries plot.")


No PM2.5 data available for timeseries plot.


In [13]:
# Distribution by site
if not pm25.empty:
    fig = viz.plot_distribution(
        pm25,
        pollutant="PM2.5",
        group_by="site",
        style="box",
        title="PM\u2082.\u2085 Distribution by AirQo Site",
    )
    plt.show()
else:
    print("No PM2.5 data for distribution plot.")


No PM2.5 data for distribution plot.


In [14]:
# Diurnal pattern
if not pm25.empty and 'PM2.5' in pm25['measurand'].values:
    fig = viz.plot_diurnal(
        pm25,
        pollutants=["PM2.5"],
        title="Diurnal PM\u2082.\u2085 Pattern (All AirQo Sites)",
    )
    plt.show()
else:
    print("No PM2.5 data for diurnal plot.")


No PM2.5 data for diurnal plot.


## Summary

This notebook demonstrated:

1. **AirQo network discovery** — exploring 200+ sensors across Africa
2. **Data quality assessment** — capture rates and ratification flags for low-cost sensors
3. **WHO compliance** — checking all interim targets (most African cities exceed even IT-1)
4. **Temporal analysis** — diurnal patterns revealing cooking and traffic emissions

### Key findings
- African urban PM\u2082.\u2085 levels are typically 5–15x the WHO annual guideline
- Data capture from low-cost sensors varies; careful QA is essential
- Diurnal patterns can reveal emission source signatures

### Next steps
- Compare dry season vs wet season (dust and biomass burning effects)
- Cross-reference with Sensor.Community sensors in the same region
- Apply correction factors for humidity effects on PM sensors